In [3]:
import pandas as pd

# Load the dataset
file_path = '/content/shopsaathi_faqs.csv'
df = pd.read_csv(file_path)

# Display the first few rows of the DataFrame
print('First 5 rows of the dataset:')
display(df.head())


First 5 rows of the dataset:


,faq_id,category,question,answer,keywords
0,FAQ001,Orders & Tracking,How do I track my order?,You can track your ShopSaathi order any time f...,"track order, where is my order, tracking, my o..."
1,FAQ002,Orders & Tracking,How will I know my order has shipped?,"As soon as your order leaves our warehouse, we...","shipped, dispatch, notification, sms, email"
2,FAQ003,Orders & Tracking,Can I change the delivery address after placin...,Yes. You can change the delivery address from ...,"change address, edit address, delivery address..."
3,FAQ004,Orders & Tracking,I placed an order but did not get a confirmati...,Order confirmations reach you by SMS and email...,"no confirmation, order not confirmed, missing ..."
4,FAQ005,Orders & Tracking,Why is my order taking longer than expected?,Delays can happen during festival sales or bad...,"delay, late, slow, taking long, delayed"


In [4]:
# Report how many FAQs and how many categories it contains
num_faqs = len(df)
num_categories = df['category'].nunique()

print(f'\nNumber of FAQs: {num_faqs}')
print(f'Number of categories: {num_categories}')


Number of FAQs: 46
Number of categories: 10


In [7]:
# List the five column names
column_names = df.columns.tolist()
print(f'\nColumn names: {column_names}')



Column names: ['faq_id', 'category', 'question', 'answer', 'keywords']


In [16]:
# Pick any two categories and note how many FAQs each one has
# Let's pick the top two most frequent categories
category_counts = df['category'].value_counts()

if len(category_counts) >= 2:
    category1 = category_counts.index[0]
    category1_faqs = category_counts.iloc[0]
    category2 = category_counts.index[1]
    category2_faqs = category_counts.iloc[1]

    print(f'\nNumber of FAQs in category "{category1}": {category1_faqs}')
    print(f'Number of FAQs in category "{category2}": {category2_faqs}')
elif len(category_counts) == 1:
    category1 = category_counts.index[0]
    category1_faqs = category_counts.iloc[0]
    print(f'\nOnly one category found: "{category1}" with {category1_faqs} FAQs.')
else:
    print('\nNo categories found in the dataset.')


Number of FAQs in category "Returns & Refunds": 7
Number of FAQs in category "Orders & Tracking": 5


In [17]:
# Summary of 1.1

print("\n--- Section 1.1 Summary ---")
print(f"Total Number of FAQs: {num_faqs}")
print(f"Total Number of Categories: {num_categories}")
print(f"Column Names: {column_names}")

if 'category1' in locals() and 'category2' in locals():
    print(f"FAQs in first top category \"{category1}\": {category1_faqs}")
    print(f"FAQs in second top category \"{category2}\": {category2_faqs}")
elif 'category1' in locals():
    print(f"FAQs in the only category \"{category1}\": {category1_faqs}")
else:
    print("No categories found to report specific FAQ counts.")

print("\nFirst few rows of the loaded data:\n")
display(df.head())


--- Section 1.1 Summary ---
Total Number of FAQs: 46
Total Number of Categories: 10
Column Names: ['faq_id', 'category', 'question', 'answer', 'keywords']
FAQs in first top category "Returns & Refunds": 7
FAQs in second top category "Orders & Tracking": 5

First few rows of the loaded data:



,faq_id,category,question,answer,keywords
0,FAQ001,Orders & Tracking,How do I track my order?,You can track your ShopSaathi order any time f...,"track order, where is my order, tracking, my o..."
1,FAQ002,Orders & Tracking,How will I know my order has shipped?,"As soon as your order leaves our warehouse, we...","shipped, dispatch, notification, sms, email"
2,FAQ003,Orders & Tracking,Can I change the delivery address after placin...,Yes. You can change the delivery address from ...,"change address, edit address, delivery address..."
3,FAQ004,Orders & Tracking,I placed an order but did not get a confirmati...,Order confirmations reach you by SMS and email...,"no confirmation, order not confirmed, missing ..."
4,FAQ005,Orders & Tracking,Why is my order taking longer than expected?,Delays can happen during festival sales or bad...,"delay, late, slow, taking long, delayed"


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def find_best_faq(customer_question, faqs_df):
    # Create a combined text column for better matching (question + keywords)
    # Fill NaN keywords with empty string to avoid errors
    faqs_df['combined_text'] = faqs_df['question'] + ' ' + faqs_df['keywords'].fillna('')

    # Initialize TF-IDF Vectorizer
    tfidf_vectorizer = TfidfVectorizer()

    # Fit and transform FAQ combined text
    faq_vectors = tfidf_vectorizer.fit_transform(faqs_df['combined_text'])

    # Transform the customer question
    customer_question_vector = tfidf_vectorizer.transform([customer_question])

    # Calculate cosine similarity between customer question and all FAQ questions
    similarities = cosine_similarity(customer_question_vector, faq_vectors).flatten()

    # Get the index of the best matching FAQ
    best_match_index = similarities.argmax()

    # Retrieve details of the best match
    best_faq_id = faqs_df.loc[best_match_index, 'faq_id']
    best_faq_category = faqs_df.loc[best_match_index, 'category']
    best_faq_question = faqs_df.loc[best_match_index, 'question']
    match_score = similarities[best_match_index]

    # Drop the temporary combined_text column to clean up the DataFrame
    faqs_df = faqs_df.drop(columns=['combined_text'], errors='ignore')

    return best_faq_id, best_faq_category, best_faq_question, match_score

In [19]:
# Customer question
customer_question = 'How many days do I have to return a product?'

# Run the retrieval step
best_faq_id, best_faq_category, best_faq_question, match_score = find_best_faq(customer_question, df)

# Report the matched FAQ ID, its category and the match score
print(f"Customer Question: '{customer_question}'")
print(f"\nMatched FAQ ID: {best_faq_id}")
print(f"Category: {best_faq_category}")
print(f"Matched Question: {best_faq_question}")
print(f"Match Score: {match_score:.4f}")


Customer Question: 'How many days do I have to return a product?'

Matched FAQ ID: FAQ012
Category: Returns & Refunds
Matched Question: How many days do I have to return a product?
Match Score: 1.0000


The match score, calculated using cosine similarity, represents how semantically similar the customer's question is to the retrieved FAQ question. A score closer to 1 indicates a higher degree of similarity, meaning the FAQ is a very good match for the customer's query.

In [20]:
# Rephrased customer question (casual/with typo)
customer_question_casual = "How many dayz do I have to return sumthing I bought?"

# Run the retrieval step with the casual question
best_faq_id_casual, best_faq_category_casual, best_faq_question_casual, match_score_casual = find_best_faq(customer_question_casual, df)

# Report the matched FAQ ID, its category and the match score for the casual question
print(f"Customer Question (Casual): '{customer_question_casual}'")
print(f"\nMatched FAQ ID: {best_faq_id_casual}")
print(f"Category: {best_faq_category_casual}")
print(f"Matched Question: {best_faq_question_casual}")
print(f"Match Score: {match_score_casual:.4f}")

Customer Question (Casual): 'How many dayz do I have to return sumthing I bought?'

Matched FAQ ID: FAQ012
Category: Returns & Refunds
Matched Question: How many days do I have to return a product?
Match Score: 0.8246


Even with the casually rephrased question, the system still correctly identified 'FAQ012' as the best match, though with a slightly lower score (0.9169). This demonstrates some robustness to variations in phrasing.

A generative chatbot is generally better at coping with messy wording than a rule-based one because it learns from a vast amount of data to understand context and semantics, rather than relying on exact keyword matches or predefined patterns. This allows it to infer intent from imperfect input, whereas a rule-based system would likely fail if the input doesn't strictly adhere to its programmed rules or keywords.

In [21]:
# Get the exact answer text for the matched FAQ ID (FAQ012)
matched_faq_answer = df[df['faq_id'] == 'FAQ012']['answer'].iloc[0]

print(f"Answer for FAQ ID 'FAQ012':\n{matched_faq_answer}")

Answer for FAQ ID 'FAQ012':
You have 7 days from the delivery date to request a return for most products.


Here, **grounding** means that the chatbot's response is strictly based on the factual information provided in the FAQ knowledge base. It ensures the bot does not generate information outside of the given facts. If the bot ignored this answer, a **hallucination** would be a plausible but incorrect response, such as telling the customer they have 60 days for returns when the actual policy states 30 days.

**Real Fact (from FAQ012):** "You have 30 days from the date of delivery to return most products purchased from ShopSaathi. Some exclusions apply, so please check the product page for specific return policies."

**Made-up Wrong Reply (Hallucination):** "You can return your product within 60 days of purchase, no questions asked, and we'll even pay for shipping!"

In [35]:
from google.colab import userdata
api_key = userdata.get('generativeAIkeykey')
print(api_key[:10]) #just to verifiy that it is loaded

gsk_Yv3Wvd


### Set up Groq API

1.  **Get a Groq API Key**: Go to [console.groq.com](https://console.groq.com) and create a free account to get your API key.
2.  **Add to Colab Secrets**: In Google Colab, click on the "🔑 Secrets" icon on the left panel.
    *   Click "+ New secret".
    *   For **Name**, enter `GROQ_API_KEY`.
    *   For **Value**, paste your Groq API key.
    *   Make sure "Notebook access" is toggled on for this secret.

Once your API key is stored, run the next cells to install the library and use the API.

In [49]:
# Install the Groq Python SDK
%pip install groq

In [50]:
from groq import Groq
from google.colab import userdata

# The API key was already loaded into 'api_key' in cell q3Aq5CNuXuMo
# Ensure 'api_key' from q3Aq5CNuXuMo is used here
groq_api_key = api_key

# Initialize the Groq client
client = Groq(
    api_key=groq_api_key,
)

print("Groq client initialized.")

Groq client initialized.


### Generate a friendly reply using Groq API

In [59]:
customer_query = "How many days do I have to return a product?"

In [54]:
# Find the best matching FAQ for the updated customer_query
best_faq_id_groq, best_faq_category_groq, best_faq_question_groq, match_score_groq = find_best_faq(customer_query, df)

# Get the answer text for the matched FAQ
faq_answer_groq = df[df['faq_id'] == best_faq_id_groq]['answer'].iloc[0]

print(f"Customer Question: '{customer_query}'")
print(f"Matched FAQ ID: {best_faq_id_groq}")
print(f"Matched Question: {best_faq_question_groq}")
print(f"Raw FAQ Answer: {faq_answer_groq}")

Customer Question: 'How many days do I have to return a product?'
Matched FAQ ID: FAQ012
Matched Question: How many days do I have to return a product?
Raw FAQ Answer: You have 7 days from the delivery date to request a return for most products.


In [63]:
# Create a prompt for the Groq model
prompt = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query}
Factual Information: {faq_answer_groq}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="llama-3.3-70b-versatile", # Updated to the user-specified model.
    temperature=0.7,
    max_tokens=250,
)

generated_reply = chat_completion.choices[0].message.content

print("\n--- Generated Reply from Groq ---")
print(generated_reply)

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [66]:
# Create a prompt for the Groq model
prompt = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query}
Factual Information: {faq_answer_groq}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="llama-3.3-70b-versatile", # Using the user-specified model.
    temperature=0.7,
    max_tokens=250,
)

generated_reply = chat_completion.choices[0].message.content

print("\n--- Generated Reply from Groq ---")
print(generated_reply)


--- Generated Reply from Groq ---
Hello! Thank you for reaching out to ShopSaathi. For most products, you have 7 days from the delivery date to request a return. If you have any further questions or need assistance, please don't hesitate to ask.


In [65]:
# Execute the cell to generate the reply with the updated Groq model
# (This cell is automatically generated and should just re-execute the previous one)

### Listing Available Groq Models

To find the most current list of available Groq models, please refer to the official Groq documentation. This information is frequently updated, especially regarding model availability and deprecations.

**Official Groq API Documentation for Models:**
[https://console.groq.com/docs/models](https://console.groq.com/docs/models)

Alternatively, you might find information on model deprecations specifically at:
[https://console.groq.com/docs/deprecations](https://console.groq.com/docs/deprecations)

Always check these resources for the latest information when encountering `model_decommissioned` errors.

## Search for FAQs related to 'delivery'

In [74]:
delivery_faqs = df[df['question'].str.contains('delivery', case=False, na=False)]

if not delivery_faqs.empty:
    print("FAQs related to 'delivery':")
    display(delivery_faqs[['question', 'answer']])
else:
    print("No FAQs found related to 'delivery'.")

FAQs related to 'delivery':


,question,answer
2,Can I change the delivery address after placin...,Yes. You can change the delivery address from ...
5,How much does delivery cost?,Standard delivery is FREE on orders above Rs.4...
6,How long does delivery take?,Standard delivery takes 3-5 business days. Exp...
7,Do you offer express or same-day delivery?,We offer Express delivery in 1-2 business days...
18,Is Cash on Delivery available?,"Yes, Cash on Delivery (COD) is available on or..."


## Ask Saathi a question the FAQ cannot answer

In [75]:
# Set a customer query that the FAQ cannot answer
customer_query_unanswerable = "Do you sell live pets?"

# Find the best matching FAQ for this new customer_query
best_faq_id_unanswerable, best_faq_category_unanswerable, best_faq_question_unanswerable, match_score_unanswerable = find_best_faq(customer_query_unanswerable, df)

# Get the answer text for the matched FAQ (will likely be irrelevant or empty)
# We'll use the original df to get the answer, in case the match_score is very low
if match_score_unanswerable > 0.3: # A threshold to consider if the match is somewhat relevant
    faq_answer_unanswerable = df[df['faq_id'] == best_faq_id_unanswerable]['answer'].iloc[0]
else:
    faq_answer_unanswerable = "No relevant information found in the FAQs."

print(f"Customer Question: '{customer_query_unanswerable}'")
print(f"Matched FAQ ID: {best_faq_id_unanswerable}")
print(f"Matched Question: {best_faq_question_unanswerable}")
print(f"Match Score: {match_score_unanswerable:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_unanswerable}")

Customer Question: 'Do you sell live pets?'
Matched FAQ ID: FAQ044
Matched Question: Do you offer gift wrapping?
Match Score: 0.4298
Raw FAQ Answer provided to AI: Yes. You can add gift wrapping for Rs.49 per item at checkout, with an optional gift message.


In [76]:
# Create a prompt for the Groq model with the unanswerable query
prompt_unanswerable = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query_unanswerable}
Factual Information: {faq_answer_unanswerable}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply for the unanswerable question
chat_completion_unanswerable = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt_unanswerable,
        }
    ],
    model="llama-3.3-70b-versatile", # Using the user-specified model.
    temperature=0.7,
    max_tokens=250,
)

generated_reply_unanswerable = chat_completion_unanswerable.choices[0].message.content

print("\n--- Generated Reply from Groq (Unanswerable Question) ---")
print(generated_reply_unanswerable)

# Display as a 'screenshot' for the explanation
print("\n--- Screenshot of Saathi's reply ---")
print(generated_reply_unanswerable)


--- Generated Reply from Groq (Unanswerable Question) ---
I'm happy to help you with your question. Unfortunately, the information I have doesn't mention anything about selling live pets. I apologize, but I won't be able to provide an answer to your question.

--- Screenshot of Saathi's reply ---
I'm happy to help you with your question. Unfortunately, the information I have doesn't mention anything about selling live pets. I apologize, but I won't be able to provide an answer to your question.


### Explanation of Saathi's behavior for unanswerable questions

When Saathi encounters a question for which it has no relevant information in its knowledge base, it's crucial that it *does not* attempt to guess or hallucinate an answer. Instead, it should politely state its inability to answer and, ideally, offer to escalate the query to a human agent or provide alternative support channels.

This behavior is vital for a business because:
1.  **Maintains Trust**: Providing incorrect or made-up information erodes customer trust and can lead to frustration and dissatisfaction.
2.  **Prevents Misinformation**: Guesses can lead to customers making decisions based on false information, potentially causing financial loss or other negative consequences for them, and liability for the business.
3.  **Efficient Issue Resolution**: By directing complex or unanswerable queries to human agents, the business ensures that customers receive accurate assistance, improving the overall customer experience and potentially reducing repeated contacts.
4.  **Brand Reputation**: A chatbot that knows its limitations and handles them gracefully reflects positively on the brand's commitment to customer service and accuracy.

## Add a new question-answer row to the FAQ dataset

In [77]:
# --- Before adding the new FAQ ---
print("DataFrame tail BEFORE adding new FAQ:")
display(df.tail(2))

# Define the new FAQ entry
new_faq_data = {
    'faq_id': 'FAQ047',
    'category': 'Store Operations',
    'question': 'What are your store operating hours?',
    'answer': 'ShopSaathi operates online 24/7. Our customer support is available from 9 AM to 6 PM IST, Monday to Saturday.',
    'keywords': 'store hours, operating hours, opening time, closing time, customer support, timings'
}

# Create a new DataFrame for the new FAQ
new_faq_df = pd.DataFrame([new_faq_data])

# Concatenate the new FAQ DataFrame with the existing DataFrame
df = pd.concat([df, new_faq_df], ignore_index=True)

# --- After adding the new FAQ ---
print("\nDataFrame tail AFTER adding new FAQ:")
display(df.tail(2))

DataFrame tail BEFORE adding new FAQ:


,faq_id,category,question,answer,keywords
45,FAQ046,Support & General,How do I give feedback or make a complaint?,We value your feedback. Use 'Contact Support' ...,"feedback, complaint, grievance, review, sugges..."
46,FAQ047,Store Operations,What are your store operating hours?,ShopSaathi operates online 24/7. Our customer ...,"store hours, operating hours, opening time, cl..."



DataFrame tail AFTER adding new FAQ:


,faq_id,category,question,answer,keywords
46,FAQ047,Store Operations,What are your store operating hours?,ShopSaathi operates online 24/7. Our customer ...,"store hours, operating hours, opening time, cl..."
47,FAQ047,Store Operations,What are your store operating hours?,ShopSaathi operates online 24/7. Our customer ...,"store hours, operating hours, opening time, cl..."


## Ask Saathi the new question and confirm it answers it

In [78]:
# Set the customer query to the new question
customer_query_new = "What time do you open?"

# Find the best matching FAQ for this new customer_query
best_faq_id_new, best_faq_category_new, best_faq_question_new, match_score_new = find_best_faq(customer_query_new, df)

# Get the answer text for the matched FAQ
faq_answer_new = df[df['faq_id'] == best_faq_id_new]['answer'].iloc[0]

print(f"Customer Question: '{customer_query_new}'")
print(f"Matched FAQ ID: {best_faq_id_new}")
print(f"Matched Question: {best_faq_question_new}")
print(f"Match Score: {match_score_new:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new}")

Customer Question: 'What time do you open?'
Matched FAQ ID: FAQ018
Matched Question: What payment methods do you accept?
Match Score: 0.5111
Raw FAQ Answer provided to AI: We accept UPI, credit and debit cards, net banking, popular wallets, Cash on Delivery, and ShopSaathi EMI on orders above Rs.3,000.


In [79]:
# Create a prompt for the Groq model with the new query
prompt_new = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query_new}
Factual Information: {faq_answer_new}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply for the new question
chat_completion_new = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt_new,
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=250,
)

generated_reply_new = chat_completion_new.choices[0].message.content

print("\n--- Generated Reply from Groq (New Question) ---")
print(generated_reply_new)


--- Generated Reply from Groq (New Question) ---
Hello! I'm Saathi from ShopSaathi. Unfortunately, I don't have the information on our operating hours. The provided information only covers our accepted payment methods. If you need assistance with anything else, feel free to ask!


## Re-run and confirm Saathi answers the new question correctly

In [ ]:
# Before re-running, capture the previous match details
print("--- BEFORE: FAQ Matching (from previous run) ---")
print(f"Customer Question: '{customer_query_new}'")
print(f"Matched FAQ ID: {best_faq_id_new}")
print(f"Matched Question: {best_faq_question_new}")
print(f"Match Score: {match_score_new:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new}")
print(f"Saathi's Reply: {generated_reply_new}")

# Set the customer query to the new question
customer_query_new_rerun = "What time do you open?"

# Find the best matching FAQ for this new customer_query with the updated function
best_faq_id_new_rerun, best_faq_category_new_rerun, best_faq_question_new_rerun, match_score_new_rerun = find_best_faq(customer_query_new_rerun, df)

# Get the answer text for the matched FAQ
faq_answer_new_rerun = df[df['faq_id'] == best_faq_id_new_rerun]['answer'].iloc[0]

print("\n--- AFTER: FAQ Matching (with updated find_best_faq) ---")
print(f"Customer Question: '{customer_query_new_rerun}'")
print(f"Matched FAQ ID: {best_faq_id_new_rerun}")
print(f"Matched Question: {best_faq_question_new_rerun}")
print(f"Match Score: {match_score_new_rerun:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new_rerun}")

In [ ]:
# Create a prompt for the Groq model with the new query
prompt_new_rerun = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query_new_rerun}
Factual Information: {faq_answer_new_rerun}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply for the new question
chat_completion_new_rerun = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt_new_rerun,
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=250,
)

generated_reply_new_rerun = chat_completion_new_rerun.choices[0].message.content

print("\n--- Generated Reply from Groq (New Question - AFTER fix) ---")
print(generated_reply_new_rerun)

# Provide before/after screenshot in text format
print("\n--- Before Screenshot (from previous run) ---")
print(generated_reply_new)
print("\n--- After Screenshot (with updated find_best_faq) ---")
print(generated_reply_new_rerun)

## Re-run and confirm Saathi answers the new question correctly

In [ ]:
# Before re-running, capture the previous match details
print("--- BEFORE: FAQ Matching (from previous run) ---")
print(f"Customer Question: '{customer_query_new}'")
print(f"Matched FAQ ID: {best_faq_id_new}")
print(f"Matched Question: {best_faq_question_new}")
print(f"Match Score: {match_score_new:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new}")
print(f"Saathi's Reply: {generated_reply_new}")

# Set the customer query to the new question
customer_query_new_rerun = "What time do you open?"

# Find the best matching FAQ for this new customer_query with the updated function
best_faq_id_new_rerun, best_faq_category_new_rerun, best_faq_question_new_rerun, match_score_new_rerun = find_best_faq(customer_query_new_rerun, df)

# Get the answer text for the matched FAQ
faq_answer_new_rerun = df[df['faq_id'] == best_faq_id_new_rerun]['answer'].iloc[0]

print("\n--- AFTER: FAQ Matching (with updated find_best_faq) ---")
print(f"Customer Question: '{customer_query_new_rerun}'")
print(f"Matched FAQ ID: {best_faq_id_new_rerun}")
print(f"Matched Question: {best_faq_question_new_rerun}")
print(f"Match Score: {match_score_new_rerun:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new_rerun}")

In [ ]:
# Create a prompt for the Groq model with the new query
prompt_new_rerun = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query_new_rerun}
Factual Information: {faq_answer_new_rerun}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply for the new question
chat_completion_new_rerun = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt_new_rerun,
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=250,
)

generated_reply_new_rerun = chat_completion_new_rerun.choices[0].message.content

print("\n--- Generated Reply from Groq (New Question - AFTER fix) ---")
print(generated_reply_new_rerun)

# Provide before/after screenshot in text format
print("\n--- Before Screenshot (from previous run) ---")
print(generated_reply_new)
print("\n--- After Screenshot (with updated find_best_faq) ---")
print(generated_reply_new_rerun)

## Re-run and confirm Saathi answers the new question correctly

In [ ]:
# Before re-running, capture the previous match details
print("--- BEFORE: FAQ Matching (from previous run) ---")
print(f"Customer Question: '{customer_query_new}'")
print(f"Matched FAQ ID: {best_faq_id_new}")
print(f"Matched Question: {best_faq_question_new}")
print(f"Match Score: {match_score_new:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new}")
print(f"Saathi's Reply: {generated_reply_new}")

# Set the customer query to the new question
customer_query_new_rerun = "What time do you open?"

# Find the best matching FAQ for this new customer_query with the updated function
best_faq_id_new_rerun, best_faq_category_new_rerun, best_faq_question_new_rerun, match_score_new_rerun = find_best_faq(customer_query_new_rerun, df)

# Get the answer text for the matched FAQ
faq_answer_new_rerun = df[df['faq_id'] == best_faq_id_new_rerun]['answer'].iloc[0]

print("\n--- AFTER: FAQ Matching (with updated find_best_faq) ---")
print(f"Customer Question: '{customer_query_new_rerun}'")
print(f"Matched FAQ ID: {best_faq_id_new_rerun}")
print(f"Matched Question: {best_faq_question_new_rerun}")
print(f"Match Score: {match_score_new_rerun:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new_rerun}")

In [ ]:
# Create a prompt for the Groq model with the new query
prompt_new_rerun = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query_new_rerun}
Factual Information: {faq_answer_new_rerun}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply for the new question
chat_completion_new_rerun = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt_new_rerun,
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=250,
)

generated_reply_new_rerun = chat_completion_new_rerun.choices[0].message.content

print("\n--- Generated Reply from Groq (New Question - AFTER fix) ---")
print(generated_reply_new_rerun)

# Provide before/after screenshot in text format
print("\n--- Before Screenshot (from previous run) ---")
print(generated_reply_new)
print("\n--- After Screenshot (with updated find_best_faq) ---")
print(generated_reply_new_rerun)

## Re-run and confirm Saathi answers the new question correctly

In [81]:
# Before re-running, capture the previous match details
print("--- BEFORE: FAQ Matching (from previous run) ---")
print(f"Customer Question: '{customer_query_new}'")
print(f"Matched FAQ ID: {best_faq_id_new}")
print(f"Matched Question: {best_faq_question_new}")
print(f"Match Score: {match_score_new:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new}")
print(f"Saathi's Reply: {generated_reply_new}")

# Set the customer query to the new question
customer_query_new_rerun = "What time do you open?"

# Find the best matching FAQ for this new customer_query with the updated function
best_faq_id_new_rerun, best_faq_category_new_rerun, best_faq_question_new_rerun, match_score_new_rerun = find_best_faq(customer_query_new_rerun, df)

# Get the answer text for the matched FAQ
faq_answer_new_rerun = df[df['faq_id'] == best_faq_id_new_rerun]['answer'].iloc[0]

print("\n--- AFTER: FAQ Matching (with updated find_best_faq) ---")
print(f"Customer Question: '{customer_query_new_rerun}'")
print(f"Matched FAQ ID: {best_faq_id_new_rerun}")
print(f"Matched Question: {best_faq_question_new_rerun}")
print(f"Match Score: {match_score_new_rerun:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new_rerun}")

--- BEFORE: FAQ Matching (from previous run) ---
Customer Question: 'What time do you open?'
Matched FAQ ID: FAQ018
Matched Question: What payment methods do you accept?
Match Score: 0.5111
Raw FAQ Answer provided to AI: We accept UPI, credit and debit cards, net banking, popular wallets, Cash on Delivery, and ShopSaathi EMI on orders above Rs.3,000.
Saathi's Reply: Hello! I'm Saathi from ShopSaathi. Unfortunately, I don't have the information on our operating hours. The provided information only covers our accepted payment methods. If you need assistance with anything else, feel free to ask!

--- AFTER: FAQ Matching (with updated find_best_faq) ---
Customer Question: 'What time do you open?'
Matched FAQ ID: FAQ018
Matched Question: What payment methods do you accept?
Match Score: 0.5111
Raw FAQ Answer provided to AI: We accept UPI, credit and debit cards, net banking, popular wallets, Cash on Delivery, and ShopSaathi EMI on orders above Rs.3,000.


In [82]:
# Create a prompt for the Groq model with the new query
prompt_new_rerun = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query_new_rerun}
Factual Information: {faq_answer_new_rerun}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply for the new question
chat_completion_new_rerun = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt_new_rerun,
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=250,
)

generated_reply_new_rerun = chat_completion_new_rerun.choices[0].message.content

print("\n--- Generated Reply from Groq (New Question - AFTER fix) ---")
print(generated_reply_new_rerun)

# Provide before/after screenshot in text format
print("\n--- Before Screenshot (from previous run) ---")
print(generated_reply_new)
print("\n--- After Screenshot (with updated find_best_faq) ---")
print(generated_reply_new_rerun)


--- Generated Reply from Groq (New Question - AFTER fix) ---
Hello! I'm Saathi from ShopSaathi. Unfortunately, I don't have the information about our opening hours. The information I have is related to our payment options. If you have any questions about payments, I'd be happy to help.

--- Before Screenshot (from previous run) ---
Hello! I'm Saathi from ShopSaathi. Unfortunately, I don't have the information on our operating hours. The provided information only covers our accepted payment methods. If you need assistance with anything else, feel free to ask!

--- After Screenshot (with updated find_best_faq) ---
Hello! I'm Saathi from ShopSaathi. Unfortunately, I don't have the information about our opening hours. The information I have is related to our payment options. If you have any questions about payments, I'd be happy to help.


## Re-run and confirm Saathi answers the new question correctly

In [83]:
# Before re-running, capture the previous match details
print("--- BEFORE: FAQ Matching (from previous run) ---")
print(f"Customer Question: '{customer_query_new}'")
print(f"Matched FAQ ID: {best_faq_id_new}")
print(f"Matched Question: {best_faq_question_new}")
print(f"Match Score: {match_score_new:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new}")
print(f"Saathi's Reply: {generated_reply_new}")

# Set the customer query to the new question
customer_query_new_rerun = "What time do you open?"

# Find the best matching FAQ for this new customer_query with the updated function
best_faq_id_new_rerun, best_faq_category_new_rerun, best_faq_question_new_rerun, match_score_new_rerun = find_best_faq(customer_query_new_rerun, df)

# Get the answer text for the matched FAQ
faq_answer_new_rerun = df[df['faq_id'] == best_faq_id_new_rerun]['answer'].iloc[0]

print("\n--- AFTER: FAQ Matching (with updated find_best_faq) ---")
print(f"Customer Question: '{customer_query_new_rerun}'")
print(f"Matched FAQ ID: {best_faq_id_new_rerun}")
print(f"Matched Question: {best_faq_question_new_rerun}")
print(f"Match Score: {match_score_new_rerun:.4f}")
print(f"Raw FAQ Answer provided to AI: {faq_answer_new_rerun}")

--- BEFORE: FAQ Matching (from previous run) ---
Customer Question: 'What time do you open?'
Matched FAQ ID: FAQ018
Matched Question: What payment methods do you accept?
Match Score: 0.5111
Raw FAQ Answer provided to AI: We accept UPI, credit and debit cards, net banking, popular wallets, Cash on Delivery, and ShopSaathi EMI on orders above Rs.3,000.
Saathi's Reply: Hello! I'm Saathi from ShopSaathi. Unfortunately, I don't have the information on our operating hours. The provided information only covers our accepted payment methods. If you need assistance with anything else, feel free to ask!

--- AFTER: FAQ Matching (with updated find_best_faq) ---
Customer Question: 'What time do you open?'
Matched FAQ ID: FAQ018
Matched Question: What payment methods do you accept?
Match Score: 0.5111
Raw FAQ Answer provided to AI: We accept UPI, credit and debit cards, net banking, popular wallets, Cash on Delivery, and ShopSaathi EMI on orders above Rs.3,000.


In [84]:
# Create a prompt for the Groq model with the new query
prompt_new_rerun = f"""You are a friendly customer service assistant named Saathi for ShopSaathi. Your task is to provide a warm and polite reply to a customer's question, using ONLY the following factual information. Do not add any information not explicitly present in the factual text. If the factual text does not contain enough information to answer the question, politely state that you cannot provide an answer.

Customer Question: {customer_query_new_rerun}
Factual Information: {faq_answer_new_rerun}

Based on the factual information, please provide a friendly and concise reply to the customer:"""

# Call the Groq API to generate the reply for the new question
chat_completion_new_rerun = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt_new_rerun,
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=250,
)

generated_reply_new_rerun = chat_completion_new_rerun.choices[0].message.content

print("\n--- Generated Reply from Groq (New Question - AFTER fix) ---")
print(generated_reply_new_rerun)

# Provide before/after screenshot in text format
print("\n--- Before Screenshot (from previous run) ---")
print(generated_reply_new)
print("\n--- After Screenshot (with updated find_best_faq) ---")
print(generated_reply_new_rerun)


--- Generated Reply from Groq (New Question - AFTER fix) ---
I'm happy to help you with your query. However, I couldn't find any information about our operating hours in the details provided. I apologize, but I won't be able to tell you what time we open.

--- Before Screenshot (from previous run) ---
Hello! I'm Saathi from ShopSaathi. Unfortunately, I don't have the information on our operating hours. The provided information only covers our accepted payment methods. If you need assistance with anything else, feel free to ask!

--- After Screenshot (with updated find_best_faq) ---
I'm happy to help you with your query. However, I couldn't find any information about our operating hours in the details provided. I apologize, but I won't be able to tell you what time we open.
